In [5]:
!pip install transformers datasets evaluate seqeval accelerate

In [6]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate

# ---------------------------------------------------------
# 1. LOAD AND CLEAN DATASET
# ---------------------------------------------------------
df = pd.read_csv("synthetic_training_data.csv")

# Clean out any empty rows (like those Dunkin ones with no city)
df = df.dropna(subset=['city'])
df['description'] = df['description'].astype(str)
df['city'] = df['city'].astype(str)

# ---------------------------------------------------------
# 2. AUTO-IOB TAGGER (Finds the city in the string)
# ---------------------------------------------------------
def create_iob_tags(row):
    desc = row['description']
    city = row['city']

    # Split the description and city into individual words
    words = str(desc).split()
    city_words = str(city).split()

    tags = ["O"] * len(words) # Default everything to "O" (Outside)

    # Find where the city words match the description words
    for i in range(len(words) - len(city_words) + 1):
        if [w.upper() for w in words[i:i+len(city_words)]] == [cw.upper() for cw in city_words]:
            tags[i] = "B-CITY" # Beginning of city
            for j in range(1, len(city_words)):
                tags[i+j] = "I-CITY" # Inside of city
            break

    return {"tokens": words, "ner_tags": tags}

# Apply the tagger and map tags to IDs
tagged_data = df.apply(create_iob_tags, axis=1)
dataset_df = pd.DataFrame(list(tagged_data))

tag2id = {"O": 0, "B-CITY": 1, "I-CITY": 2}
id2tag = {0: "O", 1: "B-CITY", 2: "I-CITY"}
dataset_df['ner_tags'] = dataset_df['ner_tags'].apply(lambda tags: [tag2id[tag] for tag in tags])

# Convert to HuggingFace Dataset format and keep 10% to test accuracy
hf_dataset = Dataset.from_pandas(dataset_df)
hf_dataset = hf_dataset.train_test_split(test_size=0.1)

# ---------------------------------------------------------
# 3. TOKENIZE FOR DISTILBERT
# ---------------------------------------------------------
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []

    for i, label in enumerate(examples[f"ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Ignore special tokens like [CLS] and [SEP]
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100) # Ignore sub-word pieces (like ##ton)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = hf_dataset.map(tokenize_and_align_labels, batched=True)

# ---------------------------------------------------------
# 4. INITIALIZE THE MODEL
# ---------------------------------------------------------
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=3,
    id2label=id2tag,
    label2id=tag2id
)

# ---------------------------------------------------------
# 5. METRICS (Accuracy, Precision, Recall, F1)
# ---------------------------------------------------------
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove the -100 padding/ignored tokens we added earlier
    true_predictions = [
        [id2tag[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2tag[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ---------------------------------------------------------
# 6. TRAINING SETUP & EXECUTION
# ---------------------------------------------------------
args = TrainingArguments(
    output_dir="./swipe_smart_ner_model",
    eval_strategy="epoch",      # Updated from evaluation_strategy to avoid deprecation warnings
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,         # 5 sweeps over your 340 rows
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  # <--- Metrics function attached here!
)

print("Starting training...")
trainer.train()

# ---------------------------------------------------------
# 7. SAVE THE CUSTOM MODEL
# ---------------------------------------------------------
trainer.save_model("./swipesmart_BERT")
print("Model trained and saved successfully! Download the 'swipe_smart_final_model' folder.")

Map:   0%|          | 0/270 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.251544,0.583333,0.280000,0.378378,0.888889
2,No log,0.047183,1.000000,1.000000,1.000000,1.000000
3,No log,0.013376,1.000000,1.000000,1.000000,1.000000
4,No log,0.007324,1.000000,1.000000,1.000000,1.000000
5,No log,0.006270,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model trained and saved successfully! Download the 'swipe_smart_final_model' folder.
